# Module 9 — Model Registry

**By the end of this notebook, you will be able to:**
- Explain why calling code should never hardcode a run ID or version number to know which model to use
- Register multiple versions of the same model under one name, using MLflow's model registry
- Use an alias to mark which version is "the one to use," and move it to a different version without touching any calling code

**Context:** Module 6 taught you to log runs; Module 7 gave you a real `Pipeline` to log. Neither tells calling code *which* logged model to actually use — you would have to hardcode a run ID or a version number, and change that hardcoded value by hand every time a better model comes along. This notebook builds the missing piece: a name that always points to "the one to use," which you can move whenever you want, without changing any code that loads it.

## Registered models have versions, not just runs

A run (Module 6) is one training/logging event. A **registered model** is a name — `toy_model` below — that groups every version ever logged under it, so you can refer to "toy_model" without caring how many times it has been retrained.

[`mlflow.sklearn.log_model(..., registered_model_name=...)`](https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html#mlflow.sklearn.log_model) does both at once: it logs the model as an MLflow artifact (exactly like Module 6's runs) and creates a new version under that name — creating the name itself, the first time it is used. Two unrelated toy models below, registered under the same name, standing in for "two versions of the same model."

In [ ]:
import mlflow
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression, Ridge

X, y = make_regression(n_samples=40, n_features=3, noise=5.0, random_state=42)

mlflow.set_tracking_uri("file:./mlruns_registry_demo")
mlflow.set_experiment("registry_demo")

with mlflow.start_run():
    info_v1 = mlflow.sklearn.log_model(
        LinearRegression().fit(X, y), name="model", registered_model_name="toy_model"
    )

with mlflow.start_run():
    info_v2 = mlflow.sklearn.log_model(
        Ridge().fit(X, y), name="model", registered_model_name="toy_model"
    )

print("version 1:", info_v1.registered_model_version)
print("version 2:", info_v2.registered_model_version)

## Stages are deprecated — use an alias instead

MLflow used to track a version's lifecycle with a fixed **stage**: `None` → `Staging` → `Production` → `Archived`. That API — `transition_model_version_stage` — has been [deprecated since MLflow 2.9](https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages), and stages are planned for removal in a future release. The current recommended practice is an **alias**: a name you attach to whichever version you choose, movable to a different version whenever you want. `champion` is the conventional alias for "the version in current use" — any name works, it is just a label.

In [ ]:
client = mlflow.MlflowClient()
client.set_registered_model_alias("toy_model", "champion", info_v1.registered_model_version)

client.get_model_version_by_alias("toy_model", "champion").version

## Load by alias, not by version

[`mlflow.sklearn.load_model`](https://mlflow.org/stable/python_api/mlflow.sklearn.html#mlflow.sklearn.load_model) accepts a `models:/<name>@<alias>` URI — calling code never needs a run ID or a version number, only the model name and the alias.

In [ ]:
champion = mlflow.sklearn.load_model("models:/toy_model@champion")
type(champion).__name__

## Move the alias, not the code

Promote version 2 instead. The exact same loading call, unchanged, now returns a different model.

In [ ]:
client.set_registered_model_alias("toy_model", "champion", info_v2.registered_model_version)

champion = mlflow.sklearn.load_model("models:/toy_model@champion")
type(champion).__name__

## Now do it for real: `air_quality`

Everything above was a toy example — two unrelated models registered under one name, with no `air_quality` code involved.

**Exercise:** Open `src/air_quality/registry.py` — `register_pipeline`, `promote_to_champion` and `load_champion` are already scaffolded, each wrapping exactly the calls you just made by hand above. Run `uv run pytest tests/test_registry.py -v` until it passes.

Then open `scripts/register_champion.py`. Its TODO rebuilds Module 7's chained `Pipeline` (`AirQualityCleaner` + `RFE` + a model) twice — once with `LinearRegression`, once with `Ridge`, Module 7's own reflection question — registers both as versions of `air_quality_model`, and promotes whichever one scores better on `cross_val_score` to `champion`. There is deliberately no fixed test for which one wins — it depends on how you built Module 7's pipeline.

In [ ]:
# Once you have run `uv run python scripts/register_champion.py`, this loads
# whichever version it promoted to champion
from air_quality.registry import load_champion

load_champion("air_quality_model")

**Continue on the site:** [Module 9 — Model Registry](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_9/)